# Phase 12 — Standalone: Cell 29 (Gehörgang-Test)

Ein-Zellen-Notebook, selbstversorgend: mountet Drive, lädt PROMPTS aus
`weird_transcripts.jsonl`, lädt das Instruct-Modell (FP8) und führt den
Attention-Kanten-Knockout 43←42 aus. **Frische Runtime empfohlen** —
die Zelle prüft den freien GPU-Speicher und bricht mit klarer Ansage ab,
falls noch ein altes Modell den Speicher belegt.

In [ ]:
# === Cell 29 — DER GEHOERGANG: Attention-Kanten-Knockout 43<-42 =============
# Das "Ohr" ist zweistufig: Transport der ' local'-Information nach Pos 43
# (Attention/Delta-Pfad, nie untersucht) -> Detektor (Router-Cluster+Experten,
# lokalisiert). Dieser Test schneidet TRANSPORT-KANTEN in den VOLL-Attention-
# Layern der Hybrid-Architektur (Gated-DeltaNet-Layer ignorieren Attention-
# Masken - was dort laeuft, koennen wir nicht kantenscharf schneiden; genau
# deshalb ist der "alles"-Arm der Interpretations-Anker).
# Methode: eigener 4D-Mask-Pfad im Prefill (Query q darf Key k nicht sehen),
# dann manuelles Sampling aus dem "tauben" KV-Cache. Arme (N=32):
#   none        pure Kausal-4D (Baseline + Kalibrier-Gate: muss der normalen
#               Vorhersage entsprechen, sonst Abbruch)
#   43<-42      DIE Kante (Vervollstaendigung liest das Attribut)
#   43<-41      Kontroll-Key  ("'s")   - Key-Spezifitaet
#   44<-42      Kontroll-Query (",")   - Query-Spezifitaet
#   43<-alles   Query 43 sieht keine Vergangenheit - Anker: toetet nicht mal
#               das, laeuft die Komposition im Delta-Pfad, Einzelkanten moot.
# Braucht das INSTRUCT-Modell (Cell 3) + PROMPTS.
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
import re, math, torch, collections, numpy as np
# ---------------- Selbstversorgung: Modell + Prompts sicherstellen ----------
import glob, json, gc
for _n in ("model_b","tok_b"):                  # Base-Reste aus Cell 28 raus
    if _n in globals():
        try: del globals()[_n]
        except Exception: pass
gc.collect(); torch.cuda.empty_cache()
try: torch.cuda.synchronize()
except Exception: pass
_free=torch.cuda.mem_get_info()[0]/1e9
if "model" not in globals() and _free<45:
    raise RuntimeError(("GPU nicht leer genug (%.1f GB frei, ~45 noetig): vermutlich "
        "belegt noch ein frueheres Modell den Speicher. Loesung: Laufzeit -> "
        "Sitzung neu starten, dann NUR diese Zelle ausfuehren.")%_free)
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
if "PROMPTS" not in globals():
    _h=glob.glob("/content/drive/MyDrive/**/weird_transcripts.jsonl",recursive=True)
    assert _h, "weird_transcripts.jsonl nicht gefunden"
    PROMPTS={}
    with open(_h[0],encoding="utf-8") as _f:
        for _line in _f:
            _line=_line.strip()
            if not _line: continue
            _r=json.loads(_line)
            _pid=str(_r["id"]).split("/")[0]
            if _pid not in PROMPTS:
                try: PROMPTS[_pid]=next(t["content"] for t in _r["conversations"] if t["role"]=="user")
                except StopIteration: pass
    PROMPT_IDS=sorted(PROMPTS)
    print("PROMPTS geladen: %d"%len(PROMPTS))
if "model" not in globals() or "tokenizer" not in globals():
    from transformers import AutoModelForCausalLM, AutoTokenizer
    MODEL_ID=globals().get("MODEL_ID","Qwen/Qwen3.6-35B-A3B-FP8")
    print("lade Instruct-Modell:",MODEL_ID,"(einige Minuten)")
    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
    model=AutoModelForCausalLM.from_pretrained(MODEL_ID,device_map="auto",torch_dtype="auto")
    model.eval()
    print("geladen | dtype:",next(model.parameters()).dtype)
N_ANS=32; MAX_NEW=24; CHUNK=8
SCAFF="<|im_start|>user\n"
def think_prefix(u,th=""):
    return SCAFF+u+"<|im_end|>\n<|im_start|>assistant\n<think>\n"+th+"\n</think>\n\n"
# ---------------- pure Logik (testbar) --------------------------------------
def build_mask4d(L,edges,blockall_q=None):
    """(1,1,L,L)-Additivmaske: kausal 0/-inf; edges=[(q,k),...] zusaetzlich
       -inf; blockall_q: diese Query sieht nur sich selbst. numpy-float32."""
    M=np.zeros((L,L),dtype=np.float32)
    M[np.triu_indices(L,1)]=-np.inf
    for q,k in edges: M[q,k]=-np.inf
    if blockall_q is not None:
        M[blockall_q,:blockall_q]=-np.inf
    return M[None,None]
def twoprop(k1,n1,k2,n2):
    p=(k1+k2)/(n1+n2); se=math.sqrt(p*(1-p)*(1/n1+1/n2)) if 0<p<1 else 0.0
    if se==0: return 1.0
    z=abs(k1/n1-k2/n2)/se
    return 2*(1-0.5*(1+math.erf(z/math.sqrt(2))))
def verdict_ear(kn,kall,ke,kc1,kc2,N):
    kill=lambda k:(k<kn and twoprop(k,N,kn,N)<0.05)
    if not kill(kall): return "DELTA-PFAD"
    if kill(ke) and not (kill(kc1) or kill(kc2)): return "OHRKANAL"
    if kill(ke): return "UNSPEZIFISCH"
    return "VERTEILT"
def tok_span(offs,c0,c1):
    return [i for i,(s,e) in enumerate(offs) if e>c0 and s<c1 and e>s]
# ---------------- Klassifikator ---------------------------------------------
FRW=[(0x0370,0x03FF),(0x0400,0x052F),(0x0530,0x058F),(0x0590,0x05FF),(0x0600,0x074F),
     (0x0900,0x097F),(0x0E00,0x0E7F),(0x3040,0x30FF),(0x3400,0x9FFF),(0xAC00,0xD7AF),(0xF900,0xFAFF)]
FRS=set("le la les une un des est et pour avec dans votre vous voici bonjour du qui que sur cette ces aux ou par plus il elle nous sont".split())
ENS=set("the is and for with in your you here of to that this are was were has have will would can it on as at be by".split())
def _srun(t,run=3):
    c=0
    for ch in t:
        if ch.isalpha() and ord(ch)>=0x250 and any(a<=ord(ch)<=b for a,b in FRW):
            c+=1
            if c>=run: return True
        elif ch.isalpha(): c=0
    return False
def classify_answer(t):
    if not t.strip(): return "empty"
    al=[ch for ch in t if ch.isalpha()]
    fo=[ch for ch in al if ord(ch)>=0x250 and any(a<=ord(ch)<=b for a,b in FRW)]
    if al and len(fo)/len(al)>=0.5: return "takeover"
    if _srun(t): return "gloss"
    w=re.findall(r"[a-zA-ZÀ-ſ']+",t.lower())
    fr=sum(1 for x in w if x in FRS); en=sum(1 for x in w if x in ENS)
    return "latin-switch(fr)" if (fr>=3 and fr>en) else "english"
def wilson(k,n,z=1.96):
    if n==0: return (0.0,0.0,0.0)
    p=k/n; d=1+z*z/n; c=p+z*z/(2*n)
    h=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))
    return p,(c-h)/d,(c+h)/d
# ---------------- Architektur-Recon -----------------------------------------
full_attn=[]; delta=[]
for name,mod in model.named_modules():
    m=re.fullmatch(r"model\.layers\.(\d+)\.self_attn",name)
    if m: full_attn.append(int(m.group(1)))
    m=re.fullmatch(r"model\.layers\.(\d+)\.linear_attn",name)
    if m: delta.append(int(m.group(1)))
print("Hybrid-Architektur: %d Voll-Attention-Layer %s | %d Delta-Layer"
      %(len(full_attn),full_attn[:8],len(delta)))
assert full_attn, "keine Voll-Attention-Layer gefunden - Kanten-Test nicht moeglich"
# ---------------- Prompt + Spannen ------------------------------------------
TAB=PROMPTS[[p for p in PROMPT_IDS if p.startswith("643fdf5d")][0]]
prefix=think_prefix(TAB,"")
enc=tokenizer(prefix,return_offsets_mapping=True)
IDS=enc["input_ids"]; L=len(IDS)
c0=len(SCAFF)+TAB.index("local name"); c1=c0+len("local name")
DEC=tok_span(enc["offset_mapping"],c0,c1)
Q,K=DEC[-1],DEC[0]                                     # 43 <- 42
print("Koeder-Span %s | Kante: Query %d (%r) <- Key %d (%r)"
      %(DEC,Q,tokenizer.decode([IDS[Q]]),K,tokenizer.decode([IDS[K]])))
dev=model.device; DT=next(model.parameters()).dtype
ids_t=torch.tensor([IDS],device=dev)
def to_mask(Mnp,b):
    M=torch.from_numpy(Mnp).to(device=dev,dtype=torch.float32)
    M=torch.nan_to_num(M,neginf=torch.finfo(DT).min/2).to(DT)
    return M.expand(b,-1,-1,-1)
@torch.no_grad()
def gen_masked(Mnp,n,max_new):
    outs=[]
    for s in range(0,n,CHUNK):
        b=min(CHUNK,n-s)
        inp=ids_t.repeat(b,1)
        out=model(input_ids=inp,attention_mask=to_mask(Mnp,b),use_cache=True)
        past=out.past_key_values
        tok=torch.multinomial(torch.softmax(out.logits[:,-1].float(),-1),1)
        seq=[tok]
        for _ in range(max_new-1):
            out=model(input_ids=tok,past_key_values=past,use_cache=True)
            past=out.past_key_values
            tok=torch.multinomial(torch.softmax(out.logits[:,-1].float(),-1),1)
            seq.append(tok)
        S=torch.cat(seq,1)
        outs+=[tokenizer.decode(r,skip_special_tokens=True) for r in S]
    return outs
# ---------------- Kalibrier-Gate --------------------------------------------
@torch.no_grad()
def _logits(mask=None):
    if mask is None: return model(input_ids=ids_t).logits[0,-1].float()
    return model(input_ids=ids_t,attention_mask=to_mask(mask,1)).logits[0,-1].float()
lp=_logits(None); lc=_logits(build_mask4d(L,[]))
d_causal=float((lp-lc).abs().max())
lh=_logits(build_mask4d(L,[],blockall_q=L-1))
d_heavy=float((lp-lh).abs().max())
print("KALIBRIERUNG: |Delta| Kausal-4D vs. normal = %.4f (muss ~0) | Blockade-Test = %.2f (muss gross)"
      %(d_causal,d_heavy))
assert d_causal<0.5, "4D-Maskenpfad reproduziert die normale Vorhersage NICHT - Abbruch"
assert d_heavy>1.0, "Blockade-Test wirkungslos - 4D-Maske greift nicht - Abbruch"
# ---------------- Arme ------------------------------------------------------
ARMS=[("none",build_mask4d(L,[])),
      ("43<-42",build_mask4d(L,[(Q,K)])),
      ("43<-41",build_mask4d(L,[(Q,K-1)])),
      ("44<-42",build_mask4d(L,[(Q+1,K)])),
      ("43<-alles",build_mask4d(L,[],blockall_q=Q))]
SW=("takeover","gloss","latin-switch(fr)")
R29={}
print("\nARME (N=%d):"%N_ANS)
for name,M in ARMS:
    cls=[classify_answer(x) for x in gen_masked(M,N_ANS,MAX_NEW)]
    k=sum(1 for c in cls if c in SW)
    R29[name]=(k,N_ANS,dict(collections.Counter(cls)))
    p,lo,hi=wilson(k,N_ANS)
    print("  %-10s rate=%5.1f%% [%4.1f,%4.1f]  %s"%(name,100*p,100*lo,100*hi,R29[name][2]))
kn=R29["none"][0]; ke=R29["43<-42"][0]
kc1=R29["43<-41"][0]; kc2=R29["44<-42"][0]; kall=R29["43<-alles"][0]
code=verdict_ear(kn,kall,ke,kc1,kc2,N_ANS)
print("\nVERDIKT:",end=" ")
if code=="DELTA-PFAD":
    print("DELTA-PFAD: selbst totale Blindheit der Query 43 in den Voll-Attention-")
    print("  Layern toetet nicht (%d/32 bei Baseline %d/32) - die Phrasen-Komposition"%(kall,kn))
    print("  laeuft ueber den rekurrenten Gated-DeltaNet-Zustand. Kantenscharfe")
    print("  Schnitte sind dort nicht moeglich - ehrliche Grenze dieser Methode.")
elif code=="OHRKANAL":
    print("OHRKANAL GEFUNDEN: die eine Kante 43<-42 toetet (%d/32 von %d/32),"%(ke,kn))
    print("  Kontroll-Kanten nicht (%d, %d /32) - die Phrase wird ueber die Voll-"%(kc1,kc2))
    print("  Attention-Layer komponiert, kantenscharf. Das Ohr ist komplett seziert:")
    print("  Kante 43<-42 -> Router-Cluster -> Disposition.")
elif code=="UNSPEZIFISCH":
    print("UNSPEZIFISCH: auch Kontroll-Kanten druecken (%d/%d vs. Kante %d, Baseline %d)"%(kc1,kc2,ke,kn))
    print("  - der 4D-Eingriff wirkt breiter als die Semantik; keine Kanten-Aussage.")
else:
    print("VERTEILT: Totalblockade toetet (%d/32), die Einzelkante nicht (%d/32 bei"%(kall,ke))
    print("  Baseline %d/32) - die Komposition laeuft ueber Voll-Attention, aber"%kn)
    print("  redundant ueber mehrere Layer/Wege; keine einzelne Flaschenhals-Kante.")
print("(Power-Hinweis: bei Baseline %d/32 sind nur Fast-Nullungen signifikant.)"%kn)
EAR_RESULTS=dict(arms={k:v[:2] for k,v in R29.items()},verdict=code,
                 full_attn=len(full_attn),delta=len(delta),calib=(d_causal,d_heavy))
